## Setup: paths, settings, model names

In [ ]:
import os, shutil, subprocess, tempfile, warnings
from itertools import combinations
from pathlib import Path

import nibabel as nib
import numpy as np
import pandas as pd
from scipy import io as sio
from scipy import stats
from scipy.linalg import orthogonal_procrustes
from scipy.spatial.distance import pdist, squareform
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.patches import Rectangle

# ---- paths (relative to this notebook's folder, like the other notebooks) ----------------------------
PROJECT  = Path("..")
CPM_DIR  = PROJECT / "4_results" / "CPM"                    # {name}_r_matrix_{Shen,Schaefer}.npy from b / b2
OUT_DIR  = PROJECT / "4_results" / "4_feature_maps"         # everything this notebook writes
SHEN_NII = PROJECT / "3_atlases" / "Shen" / "shen_2mm_268_parcellation.nii.gz"
# Schaefer NIfTI for projecting the reference models (only parcels 1-200 are used). The instructions use the
# public CBIG 17-network file; if it is not there, the 208-parcel atlas the Schaefer FC was made with is used.
SCHAEFER_NII_OPTIONS = [
    PROJECT / "3_atlases" / "downloaded" / "Schaefer2018_CBIG" / "Schaefer2018_200Parcels_17Networks_order_FSLMNI152_2mm.nii.gz",
    PROJECT / "3_atlases" / "Schaefer" / "Schaefer200_17N_plus_8ROIs_208Parcels.nii.gz",
    PROJECT / "3_atlases" / "Schaefer" / "Schaefer200_17N_plus_8ROIs_208Parcels.nii",
]
EXT_DIR   = PROJECT / "5_external_cpms"
SACPM_MAT = EXT_DIR / "saCPM_Rosenberg2016" / "saCPM.mat"
EFPM_TXT  = {s: EXT_DIR / "SurpriseEFPM_Zhang2025" / f"surprise_EFPM_{s}_mask.txt" for s in ("pos", "neg")}

# ---- settings ------------------------------------------------------------------------------------------
SIGN_RULE       = "notebook"   # sign of selected edges from np.mean (instructions); "nanmean" = sign from np.nanmean
MDS_ENGINE      = "auto"       # "auto" = R smacof if Rscript + smacof are installed, else the Python SMACOF
N_RANDOM_STARTS = 100          # random MDS starts (plus the Torgerson start)
SEED            = 20261024     # same seed as reference_space_analysis.py

ATLASES = {"Shen": 268, "Schaefer": 208}
ATLAS_TITLE = {"Shen": "Shen 268", "Schaefer": "Schaefer-Brainnetome 208"}
N_CORTICAL = 200               # Schaefer-Brainnetome ROIs 1-200 = Schaefer cortex, 201-208 = Brainnetome subcortex
AROUSAL = ["Pupil_Rest", "EEG_Rest", "Pupil_Story", "Movies"]
REFERENCE = ["saCPM", "EFPM"]
MODELS = AROUSAL + REFERENCE
LABEL = {"Pupil_Rest": "Pupil Rest", "EEG_Rest": "EEG Rest", "Pupil_Story": "Pupil Story", "Movies": "Movies",
         "saCPM": "Sustained Attention", "EFPM": "Surprise"}
ALPHAS = {"p01": 0.01, "p05": 0.05, "p10": 0.10}
MAPS = ["continuous", *ALPHAS]
MAP_TITLE = {"continuous": "Continuous", "p01": "Binarized, p < .01", "p05": "Binarized, p < .05",
             "p10": "Binarized, p < .10"}
REF_ATLAS, REF_MAP = "Shen", "p05"         # every embedding is Procrustes-aligned to this solution
FIG7_VMAX = 0.12                           # colour range of Figure 7B (+/-)
CLIP = 0.999999                            # Fisher z: r clipped to [-CLIP, CLIP]

# numbers the instructions give, checked below (a mismatch is reported, not fatal)
EXPECTED_COUNTS_P05 = {"Shen": {"Pupil_Rest": (3906, 1055), "EEG_Rest": (2843, 1209), "Pupil_Story": (2366, 1204),
                                "Movies": (4367, 1598)},
                       "Schaefer": {"Pupil_Rest": (3402, 852), "EEG_Rest": (2277, 985), "Pupil_Story": (2317, 863),
                                    "Movies": (3768, 1655)}}
EXPECTED_SHEN_P05_R = {("Pupil_Rest", "EEG_Rest"): 0.11, ("Pupil_Rest", "Pupil_Story"): 0.08,
                       ("EEG_Rest", "Pupil_Story"): 0.09, ("Pupil_Rest", "Movies"): 0.10,
                       ("EEG_Rest", "Movies"): 0.09, ("Pupil_Story", "Movies"): 0.09, ("saCPM", "EFPM"): -0.03}
EXPECTED_STRESS_P05 = {"Shen": 0.027, "Schaefer": 0.024}

# figure style (as in reference_space_analysis.py)
INK, MUTED, FRAME = "#222222", "#808080", "#c2bdb4"
BLUE, ORANGE, GREEN, DARK = "#2b78d4", "#ea6a2e", "#1a9e77", "#4d4d4d"
# arousal models: circles in the colours of the sleep/wake and 4-way network prediction figures
MODEL_STYLE = {"Pupil_Rest": ("o", "#e41a1c", 1.0), "EEG_Rest": ("o", "#377eb8", 1.0),
               "Pupil_Story": ("o", "#4daf4a", 1.0), "Movies": ("o", "#984ea3", 1.0),
               "saCPM": ("D", DARK, 0.8), "EFPM": ("+", DARK, 1.3)}
DIVERGING = matplotlib.colormaps["RdBu_r"]
CELL_EDGE = "#d4d4d4"

OUT_DIR.mkdir(parents=True, exist_ok=True)
SCHAEFER_NII = next((p for p in SCHAEFER_NII_OPTIONS if p.exists()), None)
missing = [p for p in [SHEN_NII, SACPM_MAT, *EFPM_TXT.values()] if not p.exists()]
missing += [CPM_DIR / f"{n}_r_matrix_{a}.npy" for a in ATLASES for n in AROUSAL
            if not (CPM_DIR / f"{n}_r_matrix_{a}.npy").exists()]
if SCHAEFER_NII is None:
    missing.append("a Schaefer NIfTI: " + ", ".join(map(str, SCHAEFER_NII_OPTIONS)))
if missing:
    raise FileNotFoundError("Missing inputs:\n  " + "\n  ".join(map(str, missing)))
print(f"Writing to {OUT_DIR}\nSchaefer NIfTI for the projection: {SCHAEFER_NII}")

# Functions

In [ ]:
R_SCRIPT_TEXT = r'''#!/usr/bin/env Rscript
# Interval MDS (SMACOF) for the Figure 7 reference-space embeddings.
#
# Called by reference_space_analysis.py:
#     Rscript interval_mds.R <work_dir> <n_random_starts> <seed>
#
# For every delta_<tag>.csv in <work_dir> (square dissimilarity matrix 1 - r; first column = row names),
# fits smacof::mds(delta, ndim = 2, type = "interval") from the classical (Torgerson) start and from
# <n_random_starts> random starts (set.seed(<seed>) before the random starts of each matrix), keeps the
# solution with the lowest stress-1, and writes
#     conf_<tag>.csv   2D configuration (model, x, y)
#     dhat_<tag>.csv   fitted disparities (square matrix)
#     fit_<tag>.csv    stress-1, which start won, iterations, smacof version

suppressPackageStartupMessages(library(smacof))

args <- commandArgs(trailingOnly = TRUE)
work <- args[1]
n_random <- as.integer(args[2])
seed <- as.integer(args[3])

fit_one <- function(delta, init) {
  mds(delta, ndim = 2, type = "interval", init = init, itmax = 10000, eps = 1e-12)
}

for (f in sort(list.files(work, pattern = "^delta_.*\\.csv$"))) {
  tag <- sub("^delta_(.*)\\.csv$", "\\1", f)
  D <- as.matrix(read.csv(file.path(work, f), row.names = 1, check.names = FALSE))
  delta <- as.dist(D)

  best <- fit_one(delta, "torgerson")
  best_start <- "torgerson"
  set.seed(seed)
  for (k in seq_len(n_random)) {
    fit <- fit_one(delta, "random")
    if (fit$stress < best$stress - 1e-10) {
      best <- fit
      best_start <- paste0("random_", k)
    }
  }

  conf <- best$conf
  write.csv(data.frame(model = rownames(D), x = conf[, 1], y = conf[, 2]),
            file.path(work, paste0("conf_", tag, ".csv")), row.names = FALSE)
  dhat <- as.matrix(best$dhat)
  dimnames(dhat) <- list(rownames(D), rownames(D))
  write.csv(dhat, file.path(work, paste0("dhat_", tag, ".csv")))
  write.csv(data.frame(tag = tag, stress1 = best$stress, best_start = best_start, n_iter = best$niter,
                       n_random_starts = n_random, seed = seed,
                       smacof_version = as.character(packageVersion("smacof")),
                       r_version = R.version.string),
            file.path(work, paste0("fit_", tag, ".csv")), row.names = FALSE)
}
'''


def to_matrix(v, n):
    """Edge vector (np.triu_indices(n, 1) order = itertools.combinations order) -> symmetric n x n, zero diagonal."""
    M = np.zeros((n, n))
    M[np.triu_indices(n, 1)] = v
    return M + M.T


def fisher_z(r):
    return np.arctanh(np.clip(r, -CLIP, CLIP))


def load_reference_masks():
    """Signed 268 x 268 matrices (+1 positive network, -1 negative network) and the mask checks."""
    sa = sio.loadmat(SACPM_MAT)
    raw = {"saCPM": (sa["high_attention_mask"], sa["low_attention_mask"],
                     f"{SACPM_MAT.name}: high_attention_mask / low_attention_mask"),
           "EFPM": (np.loadtxt(EFPM_TXT["pos"]), np.loadtxt(EFPM_TXT["neg"]),
                    f"{EFPM_TXT['pos'].name} / {EFPM_TXT['neg'].name}")}
    signed, rows = {}, []
    for name, (pos, neg, src) in raw.items():
        pos, neg = np.asarray(pos, float), np.asarray(neg, float)
        for part, m in (("positive", pos), ("negative", neg)):
            ok_shape = m.shape == (268, 268)
            rows.append(dict(model=name, network=part, source=src, shape=str(m.shape), symmetric=np.array_equal(m, m.T),
                             binary=bool(np.isin(m, [0, 1]).all()), zero_diagonal=bool(np.all(np.diag(m) == 0)),
                             n_edges=int(np.triu(m, 1).sum()),
                             overlap_with_other_sign=int(np.triu((pos > 0) & (neg > 0), 1).sum())))
            if not (ok_shape and rows[-1]["symmetric"] and rows[-1]["binary"] and rows[-1]["zero_diagonal"]):
                raise SystemExit(f"{name} {part} mask fails the shape/symmetry/binary check")
        if rows[-1]["overlap_with_other_sign"]:
            raise SystemExit(f"{name}: positive and negative networks overlap")
        signed[name] = pos - neg
    return signed, pd.DataFrame(rows)


def selection(r):
    """Notebook fit_model(): two-sided one-sample t-test across participants (NaN omitted). Returns p,
    np.mean (the notebooks' sign source; NaN if any participant is NaN) and np.nanmean."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        _, p = stats.ttest_1samp(r, popmean=0, axis=0, nan_policy="omit")
        mean_nanomit = np.nanmean(r, axis=0)
    p = np.ma.filled(np.ma.asarray(p, dtype=float), np.nan)
    return p, np.mean(r, axis=0), mean_nanomit


def shen_to_schaefer_weights(schaefer_nii=None):
    """W (268 x 200): W[i, j] = fraction of Shen ROI i's voxels inside Schaefer parcel j (Shen put on the
    Schaefer grid by nearest neighbour if the grids differ)."""
    sch_img, shen_img = nib.load(str(schaefer_nii or SCHAEFER_NII)), nib.load(str(SHEN_NII))
    resampled = False
    if shen_img.shape != sch_img.shape or not np.allclose(shen_img.affine, sch_img.affine):
        from nilearn.image import resample_to_img
        shen_img = resample_to_img(shen_img, sch_img, interpolation="nearest")
        resampled = True
    shen = np.rint(np.asarray(shen_img.dataobj)).astype(int)
    sch = np.rint(np.asarray(sch_img.dataobj)).astype(int)
    n_vox = np.bincount(shen[shen > 0], minlength=269)[1:]
    both = (shen > 0) & (sch > 0) & (sch <= N_CORTICAL)
    counts = np.bincount((shen[both] - 1) * N_CORTICAL + (sch[both] - 1),
                         minlength=268 * N_CORTICAL).reshape(268, N_CORTICAL)
    return counts / n_vox[:, None], resampled


def project_to_schaefer(M268, W):
    """Signed or continuous Shen matrix -> Schaefer-Brainnetome edge vector: W^T M W, zero diagonal, ROIs 1-200;
    edges touching the subcortical ROIs 201-208 get 0."""
    P = W.T @ M268 @ W
    np.fill_diagonal(P, 0)
    full = np.zeros((ATLASES["Schaefer"],) * 2)
    full[:N_CORTICAL, :N_CORTICAL] = P
    return full[np.triu_indices(ATLASES["Schaefer"], 1)]


def build_maps(R, signed, W, sign_rule):
    """maps[atlas][map] = (6, n_edges) array in MODELS order; plus per-map edge counts."""
    maps, counts = {}, []
    for atlas, n in ATLASES.items():
        E = n * (n - 1) // 2
        maps[atlas] = {m: np.zeros((len(MODELS), E)) for m in MAPS}
        for i, name in enumerate(AROUSAL):
            r = R[atlas][name]
            p, mean_nb, mean_nan = selection(r)
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", RuntimeWarning)
                cont = np.nanmean(fisher_z(r), axis=0)
            n_nan_cont = int(np.isnan(cont).sum())
            maps[atlas]["continuous"][i] = np.nan_to_num(cont)
            sign_src = mean_nb if sign_rule == "notebook" else mean_nan
            for key, a in ALPHAS.items():
                with np.errstate(invalid="ignore"):
                    sig = p < a
                keep = sig & np.isfinite(sign_src) & (sign_src != 0)
                maps[atlas][key][i] = np.where(keep, np.sign(sign_src), 0.0)
                counts.append(dict(atlas=atlas, model=name, map=key, alpha=a, n_selected=int(keep.sum()),
                                   n_positive=int((maps[atlas][key][i] > 0).sum()),
                                   n_negative=int((maps[atlas][key][i] < 0).sum()),
                                   n_p_below_alpha_without_sign=int((sig & ~keep).sum()),
                                   continuous_all_nan_edges=n_nan_cont))
        for j, ref in enumerate(REFERENCE):
            vec = signed[ref][np.triu_indices(268, 1)] if atlas == "Shen" else project_to_schaefer(signed[ref], W)
            for m in MAPS:
                maps[atlas][m][len(AROUSAL) + j] = vec
            counts.append(dict(atlas=atlas, model=ref, map="all", alpha=np.nan,
                               n_selected=int((np.abs(vec) > 1e-12).sum()), n_positive=int((vec > 1e-12).sum()),
                               n_negative=int((vec < -1e-12).sum()), n_p_below_alpha_without_sign=0,
                               continuous_all_nan_edges=0))
    return maps, pd.DataFrame(counts)


def similarity(atlas, maps_atlas):
    """Pearson r between every pair of maps (all edges): {map: dict(S=6 x 6 matrix, table=15 pairs)}."""
    out = {}
    for m, V in maps_atlas.items():
        S = np.corrcoef(V)
        table = pd.DataFrame([dict(atlas=atlas, map=m, model_a=MODELS[a], model_b=MODELS[c], label_a=LABEL[MODELS[a]],
                                   label_b=LABEL[MODELS[c]], r=S[a, c])
                              for a, c in combinations(range(len(MODELS)), 2)])
        out[m] = dict(S=S, table=table)
    return out


def torgerson(D, k=2):
    n = len(D)
    J = np.eye(n) - 1 / n
    B = -0.5 * J @ (D ** 2) @ J
    vals, vecs = np.linalg.eigh(B)
    order = np.argsort(vals)[::-1][:k]
    return vecs[:, order] * np.sqrt(np.maximum(vals[order], 0))


def smacof_python(D, kind="interval", n_random=100, max_iter=10000, tol=1e-12, seed=SEED):
    """SMACOF in 2D (Borg & Groenen 2005). kind='interval': disparities a + b * delta (b > 0) by least squares
    at each iteration, normalised to sum of squares = number of pairs (as in smacof); kind='ratio': b * delta.
    Torgerson start plus n_random random starts; returns centred configuration and stress-1
    sqrt(sum (dhat - d)^2 / sum dhat^2)."""
    n = len(D)
    dl = squareform(D, checks=False)
    A = np.column_stack([np.ones_like(dl), dl]) if kind == "interval" else dl[:, None]
    rng = np.random.default_rng(seed)
    starts = [torgerson(D)] + [rng.normal(size=(n, 2)) for _ in range(n_random)]
    best = None
    for X in starts:
        prev = np.inf
        for _ in range(max_iter):
            d = pdist(X)
            coef, *_ = np.linalg.lstsq(A, d, rcond=None)
            if coef[-1] <= 0:                                           # keep the transformation increasing
                coef = np.r_[d.mean(), 1e-9] if kind == "interval" else np.array([1e-9])
            dh = np.clip(A @ coef, 0, None)
            dh *= np.sqrt(len(dl) / np.sum(dh ** 2))
            Dh, Dm = squareform(dh), squareform(d)
            with np.errstate(divide="ignore", invalid="ignore"):
                B = np.where(Dm > 0, -Dh / Dm, 0.0)
            np.fill_diagonal(B, -B.sum(1))
            X = B @ X / n
            d = pdist(X)
            stress = np.sum((dh - d) ** 2) / np.sum(dh ** 2)
            if prev - stress < tol:
                break
            prev = stress
        if best is None or stress < best[1]:
            best = (X - X.mean(0), stress)
    return best[0], float(np.sqrt(best[1]))


def interval_mds(deltas, n_random, engine):
    """deltas: {tag: 6 x 6 DataFrame of 1 - r}. Returns {tag: (Y, info)}; Y is (6, 2) in MODELS order."""
    out = {}
    if engine == "R":
        with tempfile.TemporaryDirectory() as tmp:
            r_script = Path(tmp) / "interval_mds.R"            # the R script ships inside this notebook
            r_script.write_text(R_SCRIPT_TEXT)
            for tag, D in deltas.items():
                D.to_csv(Path(tmp) / f"delta_{tag}.csv")
            res = subprocess.run(["Rscript", str(r_script), tmp, str(n_random), str(SEED)],
                                 capture_output=True, text=True)
            if res.returncode:
                raise SystemExit(f"interval_mds.R failed:\n{res.stderr}")
            for tag, D in deltas.items():
                conf = pd.read_csv(Path(tmp) / f"conf_{tag}.csv").set_index("model").loc[MODELS]
                dhat = pd.read_csv(Path(tmp) / f"dhat_{tag}.csv", index_col=0).loc[MODELS, MODELS].to_numpy()
                fit = pd.read_csv(Path(tmp) / f"fit_{tag}.csv").iloc[0].to_dict()
                Y = conf[["x", "y"]].to_numpy()
                d, dh = pdist(Y), squareform(dhat, checks=False)
                fit["stress1_recomputed"] = float(np.sqrt(np.sum((dh - d) ** 2) / np.sum(dh ** 2)))
                fit["engine"] = "R smacof"
                out[tag] = (Y - Y.mean(0), fit)
    else:
        for tag, D in deltas.items():
            Y, s1 = smacof_python(D.to_numpy(), "interval", n_random)
            _, s1_ratio = smacof_python(D.to_numpy(), "ratio", n_random)
            out[tag] = (Y, dict(tag=tag, stress1=s1, best_start="python", n_iter=np.nan, n_random_starts=n_random,
                                seed=SEED, engine="python SMACOF", stress1_ratio_python=s1_ratio))
    return out


def orient_reference(Y):
    """Fix the arbitrary rotation/reflection of the reference solution: arousal centroid straight above the
    midpoint of the two reference models, attention (saCPM) to the left of surprise (EFPM)."""
    Y = Y - Y.mean(0)
    v = Y[:len(AROUSAL)].mean(0) - Y[len(AROUSAL):].mean(0)
    a = np.pi / 2 - np.arctan2(v[1], v[0])
    Y = Y @ np.array([[np.cos(a), np.sin(a)], [-np.sin(a), np.cos(a)]])
    if Y[MODELS.index("saCPM"), 0] > Y[MODELS.index("EFPM"), 0]:
        Y[:, 0] *= -1
    return Y


def procrustes_align(Y, Yref):
    """Rotation, reflection and uniform scaling of centred Y onto centred Yref."""
    Yc, Rc = Y - Y.mean(0), Yref - Yref.mean(0)
    Rot, sv = orthogonal_procrustes(Yc, Rc)
    scale = sv / np.sum(Yc ** 2)
    return scale * Yc @ Rot, float(scale)

In [ ]:
# ---- inputs + checks --------------------------------------------------------------------------------
R = {a: {n: np.load(CPM_DIR / f"{n}_r_matrix_{a}.npy") for n in AROUSAL} for a in ATLASES}
for a, n_roi in ATLASES.items():
    for name, r in R[a].items():
        if r.shape[1] != n_roi * (n_roi - 1) // 2:
            raise ValueError(f"{name} {a}: {r.shape[1]} columns, expected {n_roi * (n_roi - 1) // 2}")
signed, mask_df = load_reference_masks()
mask_df.to_csv(OUT_DIR / "checks_reference_masks.csv", index=False)
W, resampled = shen_to_schaefer_weights()
print(f"Reference masks OK; W {W.shape} (Shen resampled to the Schaefer grid: {resampled})")

# ---- edge maps --------------------------------------------------------------------------------------
maps, counts = build_maps(R, signed, W, SIGN_RULE)
counts.to_csv(OUT_DIR / "map_edge_counts.csv", index=False)
print("\nEdges at p < .05 (positive / negative) vs the instructions:")
for a in ATLASES:
    for name in AROUSAL:
        row = counts.query("atlas == @a and model == @name and map == 'p05'").iloc[0]
        got, exp = (row.n_positive, row.n_negative), EXPECTED_COUNTS_P05[a][name]
        wfile = CPM_DIR / f"{name}_binarized_weights_{a}_0.05.npy"
        wnote = ""
        if wfile.exists():
            w = np.load(wfile)
            wnote = ("  = b-notebook weights" if np.array_equal(np.nan_to_num(w), maps[a]["p05"][AROUSAL.index(name)])
                     else "  (differs from the b-notebook weights file)")
        print(f"  {a:9s} {name:12s} {got[0]:5d} / {got[1]:5d}   expected {exp[0]} / {exp[1]}"
              f"   {'OK' if got == exp else 'MISMATCH'}{wnote}")

# ---- similarity -------------------------------------------------------------------------------------
sim = {a: similarity(a, maps[a]) for a in ATLASES}
for a in ATLASES:
    for m in MAPS:
        pd.DataFrame(sim[a][m]["S"], index=[LABEL[x] for x in MODELS],
                     columns=[LABEL[x] for x in MODELS]).to_csv(OUT_DIR / f"similarity_{a}_{m}.csv")
S = sim["Shen"]["p05"]["S"]
print("\nShen p < .05 similarities vs the instructions:")
for (x, y), exp in EXPECTED_SHEN_P05_R.items():
    got = S[MODELS.index(x), MODELS.index(y)]
    print(f"  {LABEL[x]:20s} {LABEL[y]:20s} {got:6.3f}  expected {exp:5.2f}  {'OK' if round(got, 2) == exp else 'MISMATCH'}")

# ---- interval MDS + alignment ----------------------------------------------------------------------
engine = MDS_ENGINE
if engine == "auto":
    engine = "R" if shutil.which("Rscript") and subprocess.run(
        ["Rscript", "-e", "quit(status = !requireNamespace('smacof', quietly = TRUE))"]).returncode == 0 else "python"
print(f"\nInterval MDS engine: {engine} ({N_RANDOM_STARTS} random starts + Torgerson)")
def _delta(S):
    D = 1 - S
    np.fill_diagonal(D, 0.0)                    # (done before the DataFrame: newer pandas makes .values read-only)
    return pd.DataFrame(D, index=MODELS, columns=MODELS)


deltas = {f"{a}_{m}": _delta(sim[a][m]["S"]) for a in ATLASES for m in MAPS}
fits = interval_mds(deltas, N_RANDOM_STARTS, engine)
Yref = orient_reference(fits[f"{REF_ATLAS}_{REF_MAP}"][0])
emb, stress_rows = {}, []
for a in ATLASES:
    emb[a] = {}
    for m in MAPS:
        tag = f"{a}_{m}"
        Yraw, info = fits[tag]
        Y, scale = (Yref, 1.0) if tag == f"{REF_ATLAS}_{REF_MAP}" else procrustes_align(Yraw, Yref)
        emb[a][m] = (Y, info)
        pd.DataFrame({"model": MODELS, "label": [LABEL[x] for x in MODELS], "x": Y[:, 0], "y": Y[:, 1],
                      "x_smacof": Yraw[:, 0], "y_smacof": Yraw[:, 1]}).to_csv(OUT_DIR / f"mds_coords_{tag}.csv", index=False)
        stress_rows.append(dict(atlas=a, map=m, stress1=info["stress1"], procrustes_scale=scale,
                                aligned_to=f"{REF_ATLAS}_{REF_MAP}", best_start=info["best_start"], engine=engine))
stress_df = pd.DataFrame(stress_rows)
stress_df.to_csv(OUT_DIR / "mds_stress.csv", index=False)
print(stress_df[["atlas", "map", "stress1", "best_start"]].round(3).to_string(index=False))
for a, exp in EXPECTED_STRESS_P05.items():
    got = stress_df.query("atlas == @a and map == 'p05'")["stress1"].iloc[0]
    print(f"  {a} p < .05 stress-1 {got:.3f}, expected {exp:.3f}  {'OK' if round(got, 3) == exp else 'CHECK'}")